# FHIR on RAG

This notebook is loading FHIR resources into a vector store and then using that to help prompt an LLM to answer questions about the data. To do that, it first flattens the FHIR resources into text files. It then uses [LlamaIndex](https://www.llamaindex.ai/) to load the text files into an in-memory vector store. Then it calls out to a LLama 2 running locally using [Ollama](https://ollama.ai/) using different [strategies](https://docs.llamaindex.ai/en/stable/module_guides/querying/response_synthesizers/root.html) for combining the FHIR with the question into the prompt.

In [1]:
# Some constants to use throughout 

in_file_glob = './working/raw_fhir/*.json'
flat_file_path = './working/flat'
vector_store_file_path = './working/vector_store'

## Flatten FHIR

This is going to read in any JSON files in the `in_file_glob`. It assumes that each file is a FHIR Bundle. It will first pull out the Patient resource and extract some key information, like name, from it to include in the text files it will create per resource. This helps the RAG know which patient a resource goes with. It then flattens each resource in the bundle. 

Flattening it means that it creates a path of all the attribute names from the root of the resource to each value. In the process it splits any camel case words into multiple words. Finally, it writes this out to a text file in the structure of:
``` [path name] is [value]. ```
This creates a semi-english version of the resource that can be turned into a vector by the embedding. 

**To use this project,** you will need to create the working and raw_fhir directories and populate raw_fhir with FHIR Bundles. I used [Synthea](https://synthea.mitre.org/) to generate synthetic data in my testing.

In [2]:
import glob
import os
import json
import re

camel_pattern1 = re.compile(r'(.)([A-Z][a-z]+)')
camel_pattern2 = re.compile(r'([a-z0-9])([A-Z])')

def exclude_references(flat_entry):
    return {k: v for k, v in flat_entry.items() if 'reference' not in k.lower()}
def split_camel(text):
    new_text = camel_pattern1.sub(r'\1 \2', text)
    new_text = camel_pattern2.sub(r'\1 \2', new_text)
    return new_text


def handle_special_attributes(attrib_name, value):
    if attrib_name == 'resource Type':
        return split_camel(value)
    return value


def flatten_fhir(nested_json):
    out = {}

    def flatten(json_to_flatten, name=''):
        if type(json_to_flatten) is dict:
            json_to_flatten = exclude_references(json_to_flatten)
            for sub_attribute in json_to_flatten:
                flatten(json_to_flatten[sub_attribute], name + split_camel(sub_attribute) + ' ')
        elif type(json_to_flatten) is list:
            for i, sub_json in enumerate(json_to_flatten):
                flatten(sub_json, name + str(i) + ' ')
        else:
            attrib_name = name[:-1]
            out[attrib_name] = handle_special_attributes(attrib_name, json_to_flatten)

    flatten(nested_json)
    return out


def filter_for_patient(entry):
    return entry['resource']['resourceType'] == "Patient"


def find_patient(bundle):
    patients = list(filter(filter_for_patient, bundle['entry']))
    if len(patients) < 1:
        raise Exception('No Patient found in bundle!')
    else:
        patient = patients[0]['resource']

        patient_id = patient['id']
        first_name = patient['name'][0]['given'][0]
        last_name = patient['name'][0]['family']

        return {'PatientFirstName': first_name, 'PatientLastName': last_name, 'PatientID': patient_id}


def flat_to_string(flat_entry):
    output = ''

    for attrib in flat_entry:
        output += f'{attrib} is {flat_entry[attrib]}. '

    return output


def flatten_bundle(bundle_file_name):
    file_name = bundle_file_name[bundle_file_name.rindex('/') + 1:bundle_file_name.rindex('.')]
    with open(bundle_file_name) as raw:
        bundle = json.load(raw)
        patient = find_patient(bundle)
        flat_patient = flatten_fhir(patient)
        for i, entry in enumerate(bundle['entry']):
            flat_entry = flatten_fhir(entry['resource'])
            with open(f'{flat_file_path}/{file_name}_{i}.txt', 'w') as out_file:
                out_file.write(f'{flat_to_string(flat_patient)}\n{flat_to_string(flat_entry)}')


if not os.path.exists(flat_file_path):
    os.mkdir(flat_file_path)

for file in glob.glob(in_file_glob):
    flatten_bundle(file)

## Setup the Gen AI with RAG

This section will use LlamaIndex to construct the vector store and tie to the LLM. 

In [3]:
!pip install llama-index
!pip install transformers
!pip install llama-index-embeddings-huggingface
!pip install llama-index-llms-ollama

I tried a couple of different models for doing the embedding, i.e. turning the flattened FHIR text into vectors. I would like to experement with others, but haven't had time. In the end, `BAAI/bge-large-en-v1.5` was too big for me to run on my local, so I did most of my testing with `BAAI/bge-small-en-v1.5`.

In [4]:

from llama_index.embeddings.huggingface import HuggingFaceEmbedding

# loads BAAI/bge-small-en
# embed_model = HuggingFaceEmbedding()

embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

# embed_model = HuggingFaceEmbedding(model_name="BAAI/bge-large-en-v1.5")

# embed_model = HuggingFaceEmbedding(model_name="medicalai/ClinicalBERT")

/home/chapsk/llm/rag-fhir/RAG_on_FHIR/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [14]:
from llama_index.llms.ollama import Ollama

# LLama 2 is running locally, using Ollama.
llm = Ollama(model="llama3.2:3b-instruct-q4_0", request_timeout=300)

In [ ]:

from llama_index.core import (
    Settings,
    VectorStoreIndex,
    SimpleDirectoryReader,
    SummaryIndex # Assuming SummaryIndex is also in core, adjust if needed
)
# Assuming 'llm' and 'embed_model' are already defined LlamaIndex LLM and Embedding objects
# Configure global settings instead of using ServiceContext
Settings.llm = llm
Settings.embed_model = embed_model

# Now you can create indices or query engines, and they will use the models from Settings
# Example:
# documents = SimpleDirectoryReader(...).load_data()
# index = VectorStoreIndex.from_documents(documents) # Will use Settings.embed_model
# query_engine = index.as_query_engine() # Will use Settings.llm and Settings.embed_model


In [9]:
# This code loads the flat FHIR text files. 

documents = SimpleDirectoryReader(flat_file_path).load_data()
print(len(documents))

1124


In [10]:
# Load those flat FHIR text files into the vector store.

vector_index = VectorStoreIndex.from_documents(documents, show_progress=True)


# if not os.path.exists(vector_store_file_path):
#     os.mkdir(vector_store_file_path)
# vector_index.vector_store.persist(f'{vector_store_file_path}/FHIR_RAG.vs')

Generating embeddings: 100%|██████████| 1790/1790 [09:53<00:00,  3.01it/s]


## Actually do RAG

This is the code block that actually asks the questions of the LLM. 

In [ ]:

from llama_index.core.query_engine import RetrieverQueryEngine



# STEP 3: Create retriever and query engine
retriever = vector_index.as_retriever()
query_engine = RetrieverQueryEngine.from_args(retriever=retriever, llm=llm)

# STEP 4: Ask a question
question = "What can you tell me about Ammie189 history of hypertension?"
response = query_engine.query(question)

# STEP 5: Print the response
print("Answer:", response.response)

INFO:httpx:HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
HTTP Request: POST http://localhost:11434/api/chat "HTTP/1.1 200 OK"
Answer: Ammie189 visited an urgent care clinic on June 20, 2015, where she was seen by Dr. Jewel43 Kassulke119 for approximately 54 minutes.

She later submitted a medication request to order an outpatient treatment, which was completed on August 18, 2018, at 14:09:20.
